# 03 — Merge Factors onto the Fund Panel

| | |
|---|---|
| **Inputs** | `analysis_panel.parquet` (NB01), `factors.parquet`, `rz_monthly.parquet` (NB02) |
| **Output** | `panel_with_factors.parquet` → read by NB04–NB08 |
| **Expected result** | 373,387 fund-months, 2,138 funds, 21 columns, no missing factor values |

Fund returns are dated at month-end (`caldt`, e.g. 2000-01-31) and factors at month-start (2000-01-01), so both are
merged on a calendar year-month key. Adds `excess_ret = mret − rf`.

**To replicate:** set `DATA_ROOT` in `thesis_config.py`, run NB01 and NB02, then *Restart Kernel → Run All*.

## 0. Setup

In [ ]:
import pandas as pd

from thesis_config import DATA_DIR, assert_matches, check_paths, save_parquet

check_paths(*(DATA_DIR / f for f in ["analysis_panel.parquet", "factors.parquet", "rz_monthly.parquet"]))
print(f"pandas {pd.__version__}")

## 1. Merge

In [ ]:
panel   = pd.read_parquet(DATA_DIR / "analysis_panel.parquet")
factors = pd.read_parquet(DATA_DIR / "factors.parquet")
rz      = pd.read_parquet(DATA_DIR / "rz_monthly.parquet")
print(f"Fund panel: {panel.shape} | factors: {factors.shape} | R_Z: {rz.shape}")

panel["year_month"]   = panel["caldt"].dt.to_period("M")
factors["year_month"] = factors["date"].dt.to_period("M")

df_merged = panel.merge(factors.drop(columns="date"), on="year_month", how="left")
df_merged["excess_ret"] = df_merged["mret"] - df_merged["rf"]      # net-of-fee excess return, decimals
df_merged = df_merged.merge(rz, on="year_month", how="left")

print(f"Merged panel: {df_merged.shape}")
print(f"Columns: {df_merged.columns.tolist()}")

## 2. Integrity checks

- no row duplication and no unmatched months;
- one factor value per calendar month;
- alignment: the equal-weighted fund return must correlate more strongly with the *same* month's factor than with
  the previous or next month's (a one-month shift in the merge key would fail this);
- the fund-level mandate code carried over from NB01 is unique per fund and complete.

In [ ]:
assert len(df_merged) == len(panel), "Row count changed — merge fanned out"
for col in ["mkt_rf", "rz_monthly"]:
    assert df_merged[col].notna().all(), f"{col}: unmatched months after merge"
    assert df_merged.groupby("caldt")[col].nunique().max() <= 1, f"{col}: varies within a calendar month"
assert df_merged["excess_ret"].notna().all()

g = (df_merged.groupby("caldt")
     .agg(fund=("mret", "mean"), mkt=("mkt_rf", "first"), rz=("rz_monthly", "first"))
     .dropna())
for col in ["mkt", "rz"]:
    c = {k: g["fund"].corr(g[col].shift(k)) for k in (-1, 0, 1)}
    print(f"corr(mean fund return, {col}) at lag -1 / 0 / +1: {c[-1]:.3f} / {c[0]:.3f} / {c[1]:.3f}")
    assert abs(c[0]) > max(abs(c[-1]), abs(c[1])), f"{col}: merge appears shifted by a month"

assert df_merged["lipper_obj_cd"].notna().all()
assert df_merged.groupby("wficn")["lipper_obj_cd"].nunique().max() == 1
print("✓ All checks passed.")

## 3. Save

In [ ]:
save_parquet(df_merged, DATA_DIR / "panel_with_factors.parquet")      # read by NB04, NB05, NB08
print(f"{df_merged['wficn'].nunique():,} funds | {len(df_merged):,} fund-months | "
      f"{df_merged['caldt'].min():%Y-%m} to {df_merged['caldt'].max():%Y-%m}")
assert_matches("Sample: funds, fund-months", [df_merged["wficn"].nunique(), len(df_merged)], [2_138, 373_387], 0)